# Gram approximation error versus K: one fixed sample

Draw one reproducible sample of size $n=200$ from $N(0,1)$ and keep it fixed for every $K$. Use $\Omega=450$ and $N=50$ geometric bandwidths in $[0.01,2]$.

For this experiment, define

$$E_K=\max_{1\le r,s\le N}\left|G_{rs}-G_{rs}^{450,K}\right|.$$

The analytic reference is implemented in [density.py](../kde_tools/density.py), from the thesis's Equation `eq:beta2_exact_formula02`. The approximation uses the existing [frequency quadrature implementation](../kde_tools/fourier/similarities.py), with $K$ trapezoidal intervals and $K+1$ nodes. The same sample and bandwidth grid are used for all candidates; all Gram entries, including diagonal entries, are compared. The measured error includes both truncation and quadrature.

The seed matches the earlier fixed-sample cutoff experiment. The dashed line marks the total-error target $10^{-8}$; the dotted line marks the current production choice $K=1536$. Agreement for this one sample is an experimental result, not a uniform accuracy guarantee.

Every displayed figure is automatically saved as a PDF under `../outputs/pdfs/g_rs_error_vs_k_single_sample/`. The notebook also saves a PNG preview, the actual sample, bandwidths, settings, analytic reference matrix, and the error table under `../outputs/`.

In [ ]:
from pathlib import Path
import json
import sys

code_directory = next(
    (candidate.resolve()
     for ancestor in (Path.cwd(), *Path.cwd().parents)
     for candidate in (ancestor, ancestor / 'Code' / 'Code_kde')
     if (candidate / 'kde_tools' / 'pdf_export.py').is_file()),
    None,
)
if code_directory is None:
    raise FileNotFoundError('Run this notebook from inside the MASTERARBEIT project.')
if str(code_directory) not in sys.path:
    sys.path.insert(0, str(code_directory))

import matplotlib.pyplot as plt
import numpy as np

from kde_tools.density import exact_gaussian_kde_l2_gram
from kde_tools.fourier import (
    FOURIER_L2_METHOD_VERSION,
    gaussian_kde_l2_gram_fourier,
)
from kde_tools.pdf_export import enable_automatic_pdf_saving

notebook_name = 'g_rs_error_vs_k_single_sample'
pdf_output_directory = enable_automatic_pdf_saving(notebook_name)

In [ ]:
# One fixed N(0,1) sample; the seed matches the earlier cutoff experiment.
n = 200
seed = 20261024
N = 50
h_min, h_max = 0.01, 2.0
frequency_cutoff = 450.0
K_candidates = np.arange(64, 2304 + 1, 16, dtype=int)
highlight_K = 1536
absolute_tolerance = 1e-8

sample = np.random.default_rng(seed).normal(size=n)
bandwidths = np.geomspace(h_min, h_max, N)
data_directory = code_directory / 'outputs' / 'data' / notebook_name
image_directory = code_directory / 'outputs' / 'images' / notebook_name
data_directory.mkdir(parents=True, exist_ok=True)
image_directory.mkdir(parents=True, exist_ok=True)

settings = {
    'sample_distribution': 'standard_normal', 'realizations': 1,
    'n': n, 'seed': seed, 'N': N, 'h_min': h_min, 'h_max': h_max,
    'bandwidth_grid': 'geometric', 'frequency_cutoff': frequency_cutoff,
    'K_candidates': K_candidates.tolist(), 'highlight_K': highlight_K,
    'K_convention': 'trapezoidal intervals; frequency_points = K + 1',
    'error_definition': 'max over all r,s including diagonal of abs(G_exact - G_approx)',
    'absolute_tolerance': absolute_tolerance,
    'reference': 'exact_gaussian_kde_l2_gram_float64',
    'method_version': FOURIER_L2_METHOD_VERSION,
    'numpy_version': np.__version__,
}
np.save(data_directory / 'sample.npy', sample)
np.save(data_directory / 'bandwidths.npy', bandwidths)
(data_directory / 'settings.json').write_text(json.dumps(settings, indent=2) + '\n')
print(f'One fixed sample: n={n}, seed={seed}; Omega={frequency_cutoff:g}.')
print(f'{N} geometric bandwidths from {h_min:g} to {h_max:g}; {len(K_candidates)} K values.')

In [ ]:
exact_gram = exact_gaussian_kde_l2_gram(sample, bandwidths)
np.save(data_directory / 'exact_gram.npy', exact_gram)
errors = np.empty(len(K_candidates))
worst_indices = np.empty((len(K_candidates), 2), dtype=int)
signed_errors = np.empty(len(K_candidates))

for index, K in enumerate(K_candidates):
    approximate_gram = gaussian_kde_l2_gram_fourier(
        sample, bandwidths,
        frequency_cutoff=frequency_cutoff,
        frequency_points=int(K) + 1,
    )
    difference = approximate_gram - exact_gram
    r, s = np.unravel_index(np.argmax(np.abs(difference)), difference.shape)
    errors[index] = np.abs(difference[r, s])
    worst_indices[index] = (r, s)
    signed_errors[index] = difference[r, s]

error_table = np.column_stack([
    K_candidates, errors, worst_indices,
    bandwidths[worst_indices[:, 0]], bandwidths[worst_indices[:, 1]], signed_errors,
])
csv_path = data_directory / 'g_rs_error_vs_k.csv'
np.savetxt(
    csv_path, error_table, delimiter=',', comments='',
    header='K,E_K,worst_r_zero_based,worst_s_zero_based,h_r,h_s,approx_minus_exact_at_worst_pair',
    fmt=['%d', '%.17g', '%d', '%d', '%.17g', '%.17g', '%.17g'],
)

passing = np.flatnonzero(errors <= absolute_tolerance)
first_passing_K = int(K_candidates[passing[0]]) if len(passing) else None
highlight_index = int(np.flatnonzero(K_candidates == highlight_K)[0])
report = {
    'n': n, 'N': N, 'seed': seed, 'frequency_cutoff': frequency_cutoff,
    'first_tested_K_below_tolerance': first_passing_K,
    'absolute_tolerance': absolute_tolerance,
    'highlight_K': highlight_K, 'error_at_highlight_K': float(errors[highlight_index]),
    'max_tested_K': int(K_candidates[-1]), 'error_at_max_tested_K': float(errors[-1]),
    'minimum_observed_error': float(errors.min()),
    'minimum_error_K': int(K_candidates[np.argmin(errors)]),
    'empirical_scope': 'one fixed sample; all bandwidth pairs including diagonal',
}
(data_directory / 'report.json').write_text(json.dumps(report, indent=2) + '\n')
print(f'First tested K with E_K <= {absolute_tolerance:g}: {first_passing_K}')
print(f'E_{{{highlight_K}}} = {errors[highlight_index]:.8e}')
print(f'E_{{{K_candidates[-1]}}} = {errors[-1]:.8e}')

In [ ]:
with plt.rc_context({
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False, 'pdf.fonttype': 42,
}):
    fig, ax = plt.subplots(figsize=(8.0, 4.8), layout='constrained')
    ax.plot(K_candidates, errors, color='#2266a5', linewidth=2.0,
            label=r'$E_K=\max_{r,s}|G_{rs}-G_{rs}^{450,K}|$')
    ax.axhline(absolute_tolerance, color='#333333', linestyle='--', linewidth=1.3,
               label=r'Target $10^{-8}$')
    ax.axvline(highlight_K, color='#777777', linestyle=':', linewidth=1.2,
               label=rf'Current $K={highlight_K}$')
    ax.scatter([highlight_K], [errors[highlight_index]], s=34, color='#2266a5', zorder=3)
    ax.set_yscale('log')
    ax.set_xlim(0, K_candidates[-1])
    ax.set_xticks([0, 512, 1024, 1536, 2048])
    ax.set_xlabel(r'Number of trapezoidal intervals $K$')
    ax.set_ylabel(r'Maximum absolute Gram error $E_K$')
    ax.set_title(
        r'One fixed sample: $E_K$ versus $K$ at $\Omega=450$' + '\n'
        + rf'$n={n},\ N={N},\ h\in[{h_min:g},{h_max:g}]$; geometric bandwidth grid',
        fontsize=12, pad=12,
    )
    ax.grid(axis='y', which='major', color='#e5e5e5', linewidth=0.7)
    ax.set_axisbelow(True)
    ax.legend(loc='upper right', frameon=False, fontsize=10)
    png_path = image_directory / 'g_rs_error_vs_k.png'
    fig.savefig(png_path, dpi=200, bbox_inches='tight')
    plt.show()  # Also saves the PDF through kde_tools.pdf_export.